# gff_test

In [3]:
import gffpandas.gffpandas as gffpd

In [2]:
%pip install gffpandas

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.8/178.8 kB 2.8 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
  Created wheel for gffpandas: filename=gffpandas-1.2.0-py2.py3-none-any.whl size=6247 sha256=5459f48205f8badf0aae76c172a44e4943531aa4eaf982340b78be106e76a3d8
  Stored in directory: /gale/netapp/home/kqu/.cache/pip/wheels/bf/51/25/7409a48b5dd7683737c7729c6df69cbd976ce471fbd1f6bf0a
Successfully built gffpandas
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
df = pd.read_csv('/ceph/MethDev/clusters/ALL.CG.vs.cluster_3.bed', sep='\t', comment='t', header = None)
header = ['chromA', 'chromStartA', 'chromEndA', 'mc_A', 'chromB', 'chromStartA', 'chromEndA','mc_B']
df.columns = header[:len(df.columns)]

/tmp/ipykernel_1458023/3790910790.py:2: DtypeWarning: Columns (4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/ceph/MethDev/clusters/ALL.CG.vs.cluster_3.bed', sep='\t', comment='t', header = None)


In [18]:
df.dropna()

,chromA,chromStartA,chromEndA,mc_A,chromB,chromStartA,chromEndA,mc_B
0,chr1,108.0,109.0,0.9573,chr1,108.0,109.0,1
1,chr1,109.0,110.0,0.9215,chr1,109.0,110.0,0.878788
2,chr1,114.0,115.0,0.9362,chr1,114.0,115.0,0.941176
3,chr1,115.0,116.0,0.9369,chr1,115.0,116.0,0.971429
4,chr1,160.0,161.0,0.6764,chr1,160.0,161.0,0.741935
...,...,...,...,...,...,...,...,...
5433908,chrchrL,48412.0,48413.0,0.0000,.,-1.0,-1.0,.
5433909,chrchrL,48413.0,48414.0,0.0000,.,-1.0,-1.0,.
5433910,chrchrL,48431.0,48432.0,0.0000,.,-1.0,-1.0,.
5433911,chrchrL,48435.0,48436.0,0.0000,.,-1.0,-1.0,.


In [20]:
import pandas as pd
import pybedtools
from pybedtools import BedTool

# Load the bed files
df_path = '/ceph/MethDev/clusters/ALL.CG.vs.cluster_3.bed'
genes_path = '/ceph/MethDev/clusters/Arabidopsis_thaliana.bed'

df = pd.read_csv(df_path, sep='\t', header=None, names=['chromA', 'chromStartA', 'chromEndA', 'mc_A', 'chromB', 'chromStartB', 'chromEndB','mc_B'])
genes = pd.read_csv(genes_path, sep='\t', header=None, names=['chrom', 'start', 'end', 'info'])


In [21]:
df

,chromA,chromStartA,chromEndA,mc_A,chromB,chromStartB,chromEndB,mc_B
0,chr1,108,109,0.9573,chr1,108,109,1
1,chr1,109,110,0.9215,chr1,109,110,0.878788
2,chr1,114,115,0.9362,chr1,114,115,0.941176
3,chr1,115,116,0.9369,chr1,115,116,0.971429
4,chr1,160,161,0.6764,chr1,160,161,0.741935
...,...,...,...,...,...,...,...,...
5453616,chrPt,154297,154298,0.0000,.,-1,-1,.
5453617,chrPt,154300,154301,0.0000,.,-1,-1,.
5453618,chrPt,154322,154323,0.0000,.,-1,-1,.
5453619,chrPt,154336,154337,0.0000,.,-1,-1,.


In [22]:
genes

,chrom,start,end,info
0,chr1,3631,5899,AT1G01010
1,chr1,6788,9130,AT1G01020
2,chr1,11101,11372,AT1G03987
3,chr1,11649,13714,AT1G01030
4,chr1,23121,31227,AT1G01040
...,...,...,...,...
32828,chrPt,144921,145154,ATCG01270
32829,chrPt,145291,152175,ATCG01280
32830,chrPt,152264,152337,ATCG01290
32831,chrPt,152506,152787,ATCG01300


In [23]:
# Convert DataFrame to BedTool object
bed_df = BedTool.from_dataframe(df)
bed_genes = BedTool.from_dataframe(genes)


In [28]:
# Perform intersection
intersection = bed_df.intersect(bed_genes, wa=True, wb=True)

# Convert back to DataFrame to manipulate and analyze
intersection_df = pd.read_table(intersection.fn, header=None)
intersection_df.columns = ['chromA', 'chromStartA', 'chromEndA', 'mc_A', 'chromB', 'chromStartB', 'chromEndB','mc_B','chrom', 'start', 'end', 'info']


In [29]:
intersection_df

,chromA,chromStartA,chromEndA,mc_A,chromB,chromStartB,chromEndB,mc_B,chrom,start,end,info
0,chr1,3672,3673,0.0075,chr1,3672,3673,0.0153846,chr1,3631,5899,AT1G01010
1,chr1,3673,3674,0.0101,chr1,3673,3674,0,chr1,3631,5899,AT1G01010
2,chr1,3696,3697,0.0000,chr1,3696,3697,0,chr1,3631,5899,AT1G01010
3,chr1,3697,3698,0.0072,chr1,3697,3698,0,chr1,3631,5899,AT1G01010
4,chr1,3706,3707,0.0027,chr1,3706,3707,0,chr1,3631,5899,AT1G01010
...,...,...,...,...,...,...,...,...,...,...,...,...
3448273,chrPt,154218,154219,0.0000,.,-1,-1,.,chrPt,152806,154312,ATCG01310
3448274,chrPt,154219,154220,0.0000,.,-1,-1,.,chrPt,152806,154312,ATCG01310
3448275,chrPt,154294,154295,0.0000,.,-1,-1,.,chrPt,152806,154312,ATCG01310
3448276,chrPt,154297,154298,0.0000,.,-1,-1,.,chrPt,152806,154312,ATCG01310
